# Tokenization (mục 10 và 12.1)

Tokenizer: distilbert-base-uncased (cùng tokenizer với model ở fine_tuning.ipynb).

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text = "The cats are playing."
tokens = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
print(tokens.keys())
print("input_ids     :", tokens["input_ids"][0].tolist())
print("attention_mask:", tokens["attention_mask"][0].tolist())

KeysView({'input_ids': tensor([[ 101, 1996, 8870, 2024, 2652, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])})
input_ids     : [101, 1996, 8870, 2024, 2652, 1012, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1]


input_ids là chỉ số của từng token trong vocabulary, đây là đầu vào của lớp embedding. attention_mask đánh dấu token thật (1) và padding (0) để padding không tham gia attention như nội dung thực. Tokenizer này còn trả về token_type_ids (toàn 0 với câu đơn), DistilBERT không dùng trường này nên pretrained_classification.py chỉ truyền input_ids và attention_mask.

In [2]:
ids = tokens["input_ids"][0].tolist()
seq = tokenizer.convert_ids_to_tokens(ids)
special = [t for t, i in zip(seq, ids) if i in tokenizer.all_special_ids]
content = [t for t in seq if t not in special]
subwords = [t for t in seq if t.startswith("##")]

print("token sequence :", seq)
print("token IDs      :", ids)
print("special tokens :", special)
print("attention mask :", tokens["attention_mask"][0].tolist())
print("số words (không tính dấu câu):", len(text.replace(".", "").split()))
print("số tokens (kể cả special)    :", len(seq))
print("số tokens (không kể special) :", len(content))
print("số subword tokens (##)       :", len(subwords))

token sequence : ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
token IDs      : [101, 1996, 8870, 2024, 2652, 1012, 102]
special tokens : ['[CLS]', '[SEP]']
attention mask : [1, 1, 1, 1, 1, 1, 1]
số words (không tính dấu câu): 4
số tokens (kể cả special)    : 7
số tokens (không kể special) : 5
số subword tokens (##)       : 0


Minh họa subword và padding trong một batch có độ dài khác nhau.

Ví dụ unbelievable ở mục 5 của đề chỉ là minh họa: tokenizer này giữ nguyên unbelievable vì từ đó có trong vocabulary, còn tokenization và bertology mới bị tách thành subword. Cách tách phụ thuộc vào vocabulary của từng tokenizer.

In [3]:
for w in ["unbelievable", "playing", "cats", "tokenization", "Transformers", "bertology"]:
    print(f"{w:15} -> {tokenizer.tokenize(w)}")

batch = tokenizer(["The cats are playing.", "The cat eats fish and sleeps all day."], padding=True, return_tensors="pt")
for ids_, mask in zip(batch["input_ids"], batch["attention_mask"]):
    print(tokenizer.convert_ids_to_tokens(ids_.tolist()))
    print(mask.tolist())

unbelievable    -> ['unbelievable']
playing         -> ['playing']
cats            -> ['cats']
tokenization    -> ['token', '##ization']
Transformers    -> ['transformers']
bertology       -> ['bert', '##ology']
['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]
['[CLS]', 'the', 'cat', 'eats', 'fish', 'and', 'sleeps', 'all', 'day', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
